# 

Causal-Forest Fits on Real Data (Dictator + Trust)

Tristan Muno [](https://orcid.org/0009-0002-3078-8436) (University of Mannheim)  
Vera Okisheva (University of Mannheim)  
Raphael Klöckner (University of Mannheim)  
June 12, 2026

# Purpose

This notebook fits the **primary** estimator — `grf::causal_forest()` — on the full real data for both game outcomes. Scope is **fitting and saving only**: each forest is written to `data/03_final/` for the later post-processing step. No ATE, projection, variable importance, figures, or summaries are produced here.

The variable definitions (outcome, treatment, moderators, IDs) are reused verbatim from the validated real-data section of the estimator test `04_grf_nested_test.qmd` (`#sec-subsample`); the only change is fitting on the full per-game sample rather than a 10,000-row subsample. Respondent-level dependence across the three conjoint rounds is handled by `clusters = respondent_id`, which gives `grf` cluster-aware sampling and cluster-robust variance for free.

# Setup

In [ ]:
# execution time
start_time <- Sys.time()
# console width
options(width = 80)
# packages
p_required <- c(
  "tidyverse", # dplyr, ggplot2, tidyr
  "here", # relative paths
  "fastDummies", # country fixed-effect dummies
  "grf", # causal_forest
  "sessioninfo" # session docs
)
packages <- rownames(installed.packages())
p_to_install <- p_required[!(p_required %in% packages)]
if (length(p_to_install) > 0) {
  install.packages(p_to_install)
}
sapply(p_required, require, character.only = TRUE)


Loading required package: tidyverse

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: here

here() starts at C:/R/research/MBEU25

Loading required package: fastDummies

Loading required package: grf

Loading required package: sessioninfo

  tidyverse        here fastDummies         grf sessioninfo 
       TRUE        TRUE        TRUE        TRUE        TRUE 

# Load data

In [ ]:
dat <- readRDS(here::here("data", "02_processed", "eu25_long.rds"))


# Variable definitions

The frozen moderator list: profile-level conjoint attributes plus respondent-level pre-treatment items (items and indices together), exactly as in `04_grf_nested_test.qmd`.

In [ ]:
prof_mods <- c(
  "cj_nationality_shown",
  "der_cj_eu_identity",
  "der_cj_partisanship",
  "cj_age",
  "cj_sex",
  "cj_class"
)
resp_mods <- c(
  "q_gender",
  "q_age",
  "q_identity_country",
  "q_identity_eu",
  "q_identity_europe",
  "q_religion",
  "q_class",
  "q_eu_efficacy_understand",
  "q_pop_reps",
  "q_pop_goodevil",
  "q_pop_compromise",
  "q_dem_compromise",
  "q_dem_listen",
  "q_tech_experts",
  "q_tech_leaders",
  "q_party_harm",
  "q_people_incompetent",
  "q_eu_longterm",
  "q_eu_responsive",
  "q_eu_imp_nat_econ",
  "q_eu_imp_nat_cul",
  "q_eu_imp_nat_pol",
  "q_eu_abolish",
  "q_eu_satisfaction",
  "q_rural_urban",
  "q_edu_age_stop"
)


`prep_game()` assembles the design for one game: outcome, treatment, IDs, and the numeric moderator matrix.

In [ ]:
prep_game <- function(dat, game) {
  sub <- dat |>
    filter(cj_game_type == game) # full per-game sample (no subsampling)

  y <- sub$cj_pl2
  z <- as.integer(sub$cj_rel == "muslim") # Muslim vs all-else pooled
  country_id <- factor(sub$meta_country)
  respondent_id <- factor(sub$meta_pid)

  mod_df <- sub |>
    select(all_of(c(prof_mods, resp_mods))) |>
    mutate(across(where(is.character), factor))
  sub_X <- model.matrix(~ . - 1, data = mod_df) # factors -> dummies

  stopifnot(
    !anyNA(sub_X),
    all(z %in% c(0L, 1L)),
    !anyNA(y),
    nrow(sub_X) == length(y)
  )

  list(
    y = y,
    z = z,
    sub_X = sub_X,
    country_id = country_id,
    respondent_id = respondent_id
  )
}


`fit_grf()` fits one causal forest: moderators plus country fixed-effect dummies in `X`, known conjoint propensity, respondent clusters.

In [ ]:
fit_grf <- function(d) {
  country_fe <- fastDummies::dummy_cols(
    data.frame(country = d$country_id),
    remove_first_dummy = TRUE,
    remove_selected_columns = TRUE
  ) |>
    as.matrix()
  X <- cbind(d$sub_X, country_fe) # numeric moderators + country FE

  grf::causal_forest(
    X = X,
    Y = d$y,
    W = d$z,
    W.hat = 0.25, # known conjoint propensity
    clusters = as.integer(d$respondent_id), # respondent-level cluster-robustness
    num.trees = 2000,
    seed = 20260604
  )
}


# Dictator-game fit

In [ ]:
d_dict <- prep_game(dat, "cj_dict")
cf_dict <- fit_grf(d_dict)
saveRDS(
  cf_dict,
  here::here("data", "03_final", "grf_dictator.rds"),
  compress = "xz"
)
cat(sprintf("GRF dictator fit saved (n = %d).\n", length(d_dict$y)))


GRF dictator fit saved (n = 84603).

# Trust-game fit

In [ ]:
d_trust <- prep_game(dat, "cj_trust")
cf_trust <- fit_grf(d_trust)
saveRDS(
  cf_trust,
  here::here("data", "03_final", "grf_trust.rds"),
  compress = "xz"
)
cat(sprintf("GRF trust fit saved (n = %d).\n", length(d_trust$y)))


GRF trust fit saved (n = 84603).

# Session Info

In [ ]:
session_info()


─ Session info ───────────────────────────────────────────────────────────────
 setting  value
 version  R version 4.5.3 (2026-03-11 ucrt)
 os       Windows 11 x64 (build 26200)
 system   x86_64, mingw32
 ui       RTerm
 language (EN)
 collate  English_United States.utf8
 ctype    English_United States.utf8
 tz       Europe/Berlin
 date     2026-06-12
 pandoc   3.8.3 @ c:\\Program Files\\Positron\\resources\\app\\quarto\\bin\\tools/ (via rmarkdown)
 quarto   1.9.36 @ C:\\PROGRA~1\\Quarto\\bin\\quarto.exe

─ Packages ───────────────────────────────────────────────────────────────────
 package      * version    date (UTC) lib source
 cli            3.6.5      2025-04-23 [1] CRAN (R 4.5.1)
 codetools      0.2-20     2024-03-31 [2] CRAN (R 4.5.3)
 data.table     1.18.0     2025-12-24 [1] CRAN (R 4.5.2)
 digest         0.6.39     2025-11-19 [1] CRAN (R 4.5.3)
 dplyr        * 1.1.4      2023-11-17 [1] CRAN (R 4.5.1)
 evaluate       1.0.5      2025-08-27 [1] CRAN (R 4.5.2)
 farver         2.1

# Execution Time

In [ ]:
end_time <- Sys.time()
exec_time <- end_time - start_time
cat(paste(
  "R code execution time:",
  round(as.numeric(exec_time, units = "secs"), 2),
  "seconds."
))


R code execution time: 1478.29 seconds.

```` markdown
---
subtitle: |
  Causal-Forest Fits on Real Data (Dictator + Trust)
date: last-modified
date-format: MMMM D, YYYY
format:
  html:
    toc: true
    toc-depth: 3
    code-fold: true
    code-tools: true
execute:
  echo: true
  warning: true
  eval: true
  message: true
---

# Purpose {#sec-purpose}

This notebook fits the **primary** estimator — `grf::causal_forest()` — on the full real data for both game outcomes.
Scope is **fitting and saving only**: each forest is written to `data/03_final/` for the later post-processing step.
No ATE, projection, variable importance, figures, or summaries are produced here.

The variable definitions (outcome, treatment, moderators, IDs) are reused verbatim from the validated real-data section of the estimator test `04_grf_nested_test.qmd` (`#sec-subsample`); the only change is fitting on the full per-game sample rather than a 10,000-row subsample.
Respondent-level dependence across the three conjoint rounds is handled by `clusters = respondent_id`, which gives `grf` cluster-aware sampling and cluster-robust variance for free.

# Setup {#sec-setup}

quarto-executable-code-5450563D

```r
#| label: setup
#| include: false
# execution time
start_time <- Sys.time()
# console width
options(width = 80)
# packages
p_required <- c(
  "tidyverse", # dplyr, ggplot2, tidyr
  "here", # relative paths
  "fastDummies", # country fixed-effect dummies
  "grf", # causal_forest
  "sessioninfo" # session docs
)
packages <- rownames(installed.packages())
p_to_install <- p_required[!(p_required %in% packages)]
if (length(p_to_install) > 0) {
  install.packages(p_to_install)
}
sapply(p_required, require, character.only = TRUE)
rm(p_required, p_to_install, packages)
```

# Load data {#sec-load}

quarto-executable-code-5450563D

```r
#| label: load-data
dat <- readRDS(here::here("data", "02_processed", "eu25_long.rds"))
```

# Variable definitions {#sec-vars}

The frozen moderator list: profile-level conjoint attributes plus respondent-level pre-treatment items (items and indices together), exactly as in `04_grf_nested_test.qmd`.

quarto-executable-code-5450563D

```r
#| label: moderators
prof_mods <- c(
  "cj_nationality_shown",
  "der_cj_eu_identity",
  "der_cj_partisanship",
  "cj_age",
  "cj_sex",
  "cj_class"
)
resp_mods <- c(
  "q_gender",
  "q_age",
  "q_identity_country",
  "q_identity_eu",
  "q_identity_europe",
  "q_religion",
  "q_class",
  "q_eu_efficacy_understand",
  "q_pop_reps",
  "q_pop_goodevil",
  "q_pop_compromise",
  "q_dem_compromise",
  "q_dem_listen",
  "q_tech_experts",
  "q_tech_leaders",
  "q_party_harm",
  "q_people_incompetent",
  "q_eu_longterm",
  "q_eu_responsive",
  "q_eu_imp_nat_econ",
  "q_eu_imp_nat_cul",
  "q_eu_imp_nat_pol",
  "q_eu_abolish",
  "q_eu_satisfaction",
  "q_rural_urban",
  "q_edu_age_stop"
)
```

`prep_game()` assembles the design for one game: outcome, treatment, IDs, and the numeric moderator matrix.

quarto-executable-code-5450563D

```r
#| label: prep-game
prep_game <- function(dat, game) {
  sub <- dat |>
    filter(cj_game_type == game) # full per-game sample (no subsampling)

  y <- sub$cj_pl2
  z <- as.integer(sub$cj_rel == "muslim") # Muslim vs all-else pooled
  country_id <- factor(sub$meta_country)
  respondent_id <- factor(sub$meta_pid)

  mod_df <- sub |>
    select(all_of(c(prof_mods, resp_mods))) |>
    mutate(across(where(is.character), factor))
  sub_X <- model.matrix(~ . - 1, data = mod_df) # factors -> dummies

  stopifnot(
    !anyNA(sub_X),
    all(z %in% c(0L, 1L)),
    !anyNA(y),
    nrow(sub_X) == length(y)
  )

  list(
    y = y,
    z = z,
    sub_X = sub_X,
    country_id = country_id,
    respondent_id = respondent_id
  )
}
```

`fit_grf()` fits one causal forest: moderators plus country fixed-effect dummies in `X`, known conjoint propensity, respondent clusters.

quarto-executable-code-5450563D

```r
#| label: fit-grf
fit_grf <- function(d) {
  country_fe <- fastDummies::dummy_cols(
    data.frame(country = d$country_id),
    remove_first_dummy = TRUE,
    remove_selected_columns = TRUE
  ) |>
    as.matrix()
  X <- cbind(d$sub_X, country_fe) # numeric moderators + country FE

  grf::causal_forest(
    X = X,
    Y = d$y,
    W = d$z,
    W.hat = 0.25, # known conjoint propensity
    clusters = as.integer(d$respondent_id), # respondent-level cluster-robustness
    num.trees = 2000,
    seed = 20260604
  )
}
```

# Dictator-game fit {#sec-dictator}

quarto-executable-code-5450563D

```r
#| label: fit-dictator
d_dict <- prep_game(dat, "cj_dict")
cf_dict <- fit_grf(d_dict)
saveRDS(
  cf_dict,
  here::here("data", "03_final", "grf_dictator.rds"),
  compress = "xz"
)
cat(sprintf("GRF dictator fit saved (n = %d).\n", length(d_dict$y)))
```

# Trust-game fit {#sec-trust}

quarto-executable-code-5450563D

```r
#| label: fit-trust
d_trust <- prep_game(dat, "cj_trust")
cf_trust <- fit_grf(d_trust)
saveRDS(
  cf_trust,
  here::here("data", "03_final", "grf_trust.rds"),
  compress = "xz"
)
cat(sprintf("GRF trust fit saved (n = %d).\n", length(d_trust$y)))
```

# Session Info {#sec-session-info .appendix .unnumbered}

quarto-executable-code-5450563D

```r
#| label: session-info
#| eval: true
#| echo: true
session_info()
```

# Execution Time {#sec-exec-time .appendix .unnumbered}

quarto-executable-code-5450563D

```r
#| label: exec-time
#| eval: true
#| echo: true
#| include: true
end_time <- Sys.time()
exec_time <- end_time - start_time
cat(paste(
  "R code execution time:",
  round(as.numeric(exec_time, units = "secs"), 2),
  "seconds."
))
```
````